In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, Birch
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from scipy.stats import ttest_ind, chi2_contingency
import matplotlib.pyplot as plt

# Load dataset
file_path = 'Study_Dataset.csv'
df = pd.read_csv(file_path)

demographic_cols = ['ADI_NATRANK', 'adi_staterank',
    'demographics_HealthCoverage_MediCal', 'demographics_HealthCoverage_Medicare',
    'demographics_is_age_gte78_lt83', 'demographics_is_age_gte83_lt88',
    'demographics_is_age_gte88_lt93', 'demographics_is_age_gte93_lt98',
    'demographics_is_pat_sex_female', 'demographics_is_race_asian', 'demographics_is_race_black',
    'demographics_is_race_native_american', 'demographics_is_race_other', 'demographics_is_race_unknown',
    'demographics_is_race_white', 'demographics_Managed_Care_Anthem_Blue_Cross',
    'demographics_Managed_Care_Blue_Shield_HMO', 'demographics_Managed_Care_Cigna_HMO',
    'demographics_Managed_Care_CMS_MSSP', 'demographics_Managed_Care_Health_Net_Blue_Gold',
    'demographics_Managed_Care_Health_Net_GMC', 'demographics_Managed_Care_Health_Net_Senior',
    'demographics_Managed_Care_United_Health_Care', 'demographics_Managed_Care_Western_Health_Advantage',
    'demographics_pat_age']

# Split data
df_0 = df[df['OUTCOME'] == 0][demographic_cols]
df_1 = df[df['OUTCOME'] == 1][demographic_cols]

# Normalize
scaler = StandardScaler()
df_0_scaled = scaler.fit_transform(df_0)
df_1_scaled = scaler.fit_transform(df_1)

# Clustering models
agglom_0 = AgglomerativeClustering(n_clusters=3)
labels_0 = agglom_0.fit_predict(df_0_scaled)

kmeans_1 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_1 = kmeans_1.fit_predict(df_1_scaled)

# Assign labels
df_0_clustered = df_0.copy()
df_0_clustered['Cluster'] = labels_0

df_1_clustered = df_1.copy()
df_1_clustered['Cluster'] = labels_1

# Means per cluster
print("Mean demographics for OUTCOME=0")
print(df_0_clustered.groupby('Cluster').mean())

print("\nMean demographics for OUTCOME=1")
print(df_1_clustered.groupby('Cluster').mean())

# Statistical tests
def compare_clusters_ttest(df1, df2, continuous_cols):
    results = {}
    for col in continuous_cols:
        stat, pval = ttest_ind(df1[col], df2[col], equal_var=False)
        results[col] = {'t_stat': stat, 'p_value': pval}
    return pd.DataFrame(results).T

def compare_clusters_chi2(df):
    results = {}
    categorical_cols = [col for col in demographic_cols if col not in ['ADI_NATRANK','adi_staterank','demographics_pat_age']]
    for col in categorical_cols:
        table = pd.crosstab(df['Cluster'], df[col])
        if table.shape[1] == 1:
            continue
        chi2, p, _, _ = chi2_contingency(table)
        results[col] = {'chi2_stat': chi2, 'p_value': p}
    return pd.DataFrame(results).T

# Within OUTCOME=0
print("\n--- Stats within OUTCOME=0 ---")
for pair in [(0,1),(0,2),(1,2)]:
    df_a = df_0_clustered[df_0_clustered['Cluster']==pair[0]]
    df_b = df_0_clustered[df_0_clustered['Cluster']==pair[1]]
    print(f"\nT-test {pair[0]} vs {pair[1]}:")
    print(compare_clusters_ttest(df_a, df_b, ['ADI_NATRANK','adi_staterank','demographics_pat_age']))
    print(f"\nChi-squared {pair[0]} vs {pair[1]}:")
    print(compare_clusters_chi2(pd.concat([df_a, df_b])))

# Within OUTCOME=1
print("\n--- Stats within OUTCOME=1 ---")
for pair in [(0,1),(0,2),(1,2)]:
    df_a = df_1_clustered[df_1_clustered['Cluster']==pair[0]]
    df_b = df_1_clustered[df_1_clustered['Cluster']==pair[1]]
    print(f"\nT-test {pair[0]} vs {pair[1]}:")
    print(compare_clusters_ttest(df_a, df_b, ['ADI_NATRANK','adi_staterank','demographics_pat_age']))
    print(f"\nChi-squared {pair[0]} vs {pair[1]}:")
    print(compare_clusters_chi2(pd.concat([df_a, df_b])))

# Between same-number clusters
print("\n--- Stats between OUTCOME=0 and 1 by same cluster id ---")
for cid in [0,1,2]:
    df0c = df_0_clustered[df_0_clustered['Cluster']==cid]
    df1c = df_1_clustered[df_1_clustered['Cluster']==cid]
    print(f"\nT-test Cluster {cid} 0 vs 1:")
    print(compare_clusters_ttest(df0c, df1c, ['ADI_NATRANK','adi_staterank','demographics_pat_age']))
    print(f"\nChi-squared Cluster {cid} 0 vs 1:")
    print(compare_clusters_chi2(pd.concat([df0c, df1c])))


Mean demographics for OUTCOME=0
         ADI_NATRANK  adi_staterank  demographics_HealthCoverage_MediCal  \
Cluster                                                                    
0          17.735465       5.444767                             0.000000   
1          18.361746       5.767152                             0.000000   
2          18.751489       5.680851                             0.012766   

         demographics_HealthCoverage_Medicare  demographics_is_age_gte78_lt83  \
Cluster                                                                         
0                                    0.136628                        0.005814   
1                                    0.831601                        0.149688   
2                                    0.011915                        0.000000   

         demographics_is_age_gte83_lt88  demographics_is_age_gte88_lt93  \
Cluster                                                                   
0                              